# Advanced SIEM CSV Analysis

This notebook performs a full exploratory analysis of `advanced_siem.csv`.

Each section follows the same pattern:
- First, markdown explains what the next block is doing and why it matters.
- Then, the code cell implements that step with inline comments.

The analysis focuses on:
- dataset structure and data quality
- timestamp behavior and time-based patterns
- event, severity, and source distributions
- field usage by event type
- parsed metadata such as risk score, confidence, and behavioral analytics
- security-oriented findings and investigation priorities

The notebook uses only Python standard-library modules so it can run without extra package installation.

## 1. Load the CSV and build reusable helper functions

This section loads the dataset, converts timestamps into Python `datetime` objects, and defines a few small helpers.

Why this comes first:
- every later section depends on a clean in-memory representation of the rows
- helper functions keep the rest of the notebook shorter and easier to read
- parsing is handled carefully because several columns contain dictionary-like strings

In [ ]:
import csv
import ast
import math
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from statistics import mean

DATA_PATH = Path('advanced_siem.csv')

def is_missing(value):
    # Treat blanks and placeholder values as missing for profiling.
    return value in ('', None, 'N/A')

def parse_mapping(value):
    # Some columns store Python-style dict strings; convert them safely.
    if is_missing(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}

def to_float(value):
    # Convert numeric strings when possible and return None otherwise.
    if is_missing(value):
        return None
    try:
        return float(value)
    except ValueError:
        return None

def pct(part, whole):
    return round((part / whole) * 100, 2) if whole else 0.0

def top_items(counter_obj, limit=10):
    return counter_obj.most_common(limit)

with DATA_PATH.open(newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row['parsed_timestamp'] = datetime.strptime(row['timestamp'], '%Y-%m-%d %H:%M:%S')
    row['advanced_metadata_dict'] = parse_mapping(row['advanced_metadata'])
    row['behavioral_analytics_dict'] = parse_mapping(row['behavioral_analytics'])

column_names = list(rows[0].keys())
original_columns = [name for name in column_names if not name.endswith('_dict') and name != 'parsed_timestamp']

print(f'Rows: {len(rows):,}')
print(f'Columns in CSV: {len(original_columns)}')
print('First 10 columns:', original_columns[:10])


## 2. Inspect the schema and sample records

Before computing statistics, it is useful to confirm what the table looks like and what kinds of fields each record contains.

This step answers:
- which columns exist
- what a typical record looks like
- whether the dataset appears multi-domain instead of a single log source

In [ ]:
# Print the schema so the notebook reader can see every available field.
print('Schema:')
for index, name in enumerate(original_columns, start=1):
    print(f'{index:>2}. {name}')

print('\nSample records (trimmed to a few high-signal columns):')
sample_fields = ['timestamp', 'event_type', 'source', 'severity', 'user', 'action', 'src_ip', 'alert_type', 'device_type', 'cloud_service', 'model_id']
for row in rows[:5]:
    trimmed = {field: row[field] for field in sample_fields}
    print(trimmed)


## 3. Check data quality and missingness

A complete analysis should measure how much of each field is actually populated.

This is important because many SIEM datasets are schema-rich but sparsely populated depending on the event type. A column may exist globally while being meaningful only for one subtype of event.

In [ ]:
missing_counts = Counter()
non_missing_counts = Counter()

for row in rows:
    for column in original_columns:
        if is_missing(row[column]):
            missing_counts[column] += 1
        else:
            non_missing_counts[column] += 1

quality_summary = []
for column in original_columns:
    present = non_missing_counts[column]
    missing = missing_counts[column]
    quality_summary.append((column, present, missing, pct(present, len(rows))))

quality_summary.sort(key=lambda item: item[2], reverse=True)

print('Columns with the most missing values:')
for column, present, missing, present_pct in quality_summary[:20]:
    print(f'{column:<22} present={present:>6,} missing={missing:>6,} present_pct={present_pct:>6.2f}%')

print('\nColumns that are fully populated:')
for column, present, missing, present_pct in quality_summary:
    if missing == 0:
        print(f'{column} ({present_pct:.2f}% present)')


## 4. Understand the time range and timestamp quality

This section checks the temporal coverage of the dataset and looks for unusual timestamp behavior.

In security analysis, time quality matters because alert spikes, attack chains, and incident timelines are all time-dependent. If the timestamp field contains out-of-range values, trend analysis can be misleading.

In [ ]:
timestamps = [row['parsed_timestamp'] for row in rows]
min_ts = min(timestamps)
max_ts = max(timestamps)

by_month = Counter(row['parsed_timestamp'].strftime('%Y-%m') for row in rows)
by_weekday = Counter(row['parsed_timestamp'].strftime('%A') for row in rows)
by_hour = Counter(row['parsed_timestamp'].hour for row in rows)

# Use 2025 as the expected core period because it dominates the dataset.
out_of_2025 = [row for row in rows if row['parsed_timestamp'].year != 2025]

print('Timestamp range:')
print('Min:', min_ts)
print('Max:', max_ts)
print(f'Rows outside 2025: {len(out_of_2025):,} ({pct(len(out_of_2025), len(rows)):.2f}%)')

print('\nTop months:')
for month, count in top_items(by_month, 12):
    print(f'{month}: {count:,}')

print('\nWeekday distribution:')
for day, count in by_weekday.most_common():
    print(f'{day:<10} {count:>6,}')

print('\nTop hours:')
for hour, count in top_items(by_hour, 10):
    print(f'{hour:02d}:00 -> {count:,}')


## 5. Measure high-level event, severity, and source distributions

This section gives the main shape of the dataset.

It answers:
- which event families are most common
- how severe the records are overall
- which SIEM or security products appear most frequently as data sources

In [ ]:
event_type_counts = Counter(row['event_type'] for row in rows)
severity_counts = Counter(row['severity'] for row in rows)
source_counts = Counter(row['source'] for row in rows)

print('Event types:')
for name, count in event_type_counts.most_common():
    print(f'{name:<12} {count:>6,} ({pct(count, len(rows)):>6.2f}%)')

print('\nSeverity levels:')
for name, count in severity_counts.most_common():
    print(f'{name:<12} {count:>6,} ({pct(count, len(rows)):>6.2f}%)')

print('\nTop sources:')
for name, count in top_items(source_counts, 15):
    print(f'{name:<28} {count:>6,}')


## 6. See which fields are used by each event type

The missing-value profile becomes more useful when broken down by event family.

This section shows that the dataset is not randomly sparse. Instead, each event type populates a different subset of columns. That is a key modeling insight if you later build parsers, dashboards, or anomaly detectors.

In [ ]:
focus_fields = [
    'user', 'action', 'object', 'process_id', 'parent_process', 'device_type',
    'src_ip', 'alert_type', 'category', 'cloud_service', 'model_id',
    'protocol', 'bytes', 'duration'
]

usage_by_event_type = defaultdict(Counter)
event_totals = Counter(row['event_type'] for row in rows)

for row in rows:
    event_type = row['event_type']
    for field in focus_fields:
        if not is_missing(row[field]):
            usage_by_event_type[event_type][field] += 1

for event_type in sorted(usage_by_event_type):
    print(f'\nEvent type: {event_type} (rows={event_totals[event_type]:,})')
    for field in focus_fields:
        count = usage_by_event_type[event_type][field]
        print(f'  {field:<15} {count:>6,} ({pct(count, event_totals[event_type]):>6.2f}%)')


## 7. Analyze severity by event type

A flat severity distribution can hide important differences between event categories.

This section creates a severity matrix so we can answer questions such as:
- which event type produces emergency events
- which families skew toward critical findings
- whether some event classes are mostly informational

In [ ]:
severity_matrix = defaultdict(Counter)
for row in rows:
    severity_matrix[row['event_type']][row['severity']] += 1

severity_order = ['emergency', 'critical', 'high', 'medium', 'low', 'info']

for event_type in sorted(severity_matrix):
    print(f'\nEvent type: {event_type}')
    for severity in severity_order:
        count = severity_matrix[event_type][severity]
        if count:
            print(f'  {severity:<10} {count:>6,} ({pct(count, event_totals[event_type]):>6.2f}%)')


## 8. Parse `advanced_metadata` for risk, confidence, and geography

The `advanced_metadata` column contains structured enrichment data embedded as a string. This is one of the most valuable fields in the dataset because it adds context beyond the base log line.

The code below extracts:
- `risk_score`
- `confidence`
- `geo_location`

These values help estimate investigation priority and assess how enriched the events are.

In [ ]:
risk_scores = []
confidence_scores = []
geo_locations = Counter()

for row in rows:
    meta = row['advanced_metadata_dict']
    risk = to_float(meta.get('risk_score'))
    confidence = to_float(meta.get('confidence'))

    if risk is not None:
        risk_scores.append(risk)
    if confidence is not None:
        confidence_scores.append(confidence)
    if meta.get('geo_location'):
        geo_locations[meta['geo_location']] += 1

print('Risk score summary:')
print(f'  count={len(risk_scores):,} avg={mean(risk_scores):.2f} min={min(risk_scores):.2f} max={max(risk_scores):.2f}')

print('Confidence summary:')
print(f'  count={len(confidence_scores):,} avg={mean(confidence_scores):.3f} min={min(confidence_scores):.2f} max={max(confidence_scores):.2f}')

print('\nTop geolocations:')
for place, count in top_items(geo_locations, 15):
    print(f'{place:<20} {count:>6,}')


## 9. Parse `behavioral_analytics` where available

This field is populated for only a subset of records, but it still deserves separate analysis.

Behavioral analytics often contains high-value anomaly indicators such as deviation scores, entropy, and boolean anomaly flags. Even partial coverage can be useful if those records are concentrated in specific event classes.

In [ ]:
baseline_deviations = []
entropy_scores = []
frequency_anomaly_true = 0
sequence_anomaly_true = 0
behavioral_rows = 0

for row in rows:
    details = row['behavioral_analytics_dict']
    if not details:
        continue

    behavioral_rows += 1
    baseline = to_float(details.get('baseline_deviation'))
    entropy = to_float(details.get('entropy'))

    if baseline is not None:
        baseline_deviations.append(baseline)
    if entropy is not None:
        entropy_scores.append(entropy)
    if details.get('frequency_anomaly') is True:
        frequency_anomaly_true += 1
    if details.get('sequence_anomaly') is True:
        sequence_anomaly_true += 1

print(f'Rows with behavioral analytics: {behavioral_rows:,} ({pct(behavioral_rows, len(rows)):.2f}%)')
print(f'Average baseline deviation: {mean(baseline_deviations):.3f}')
print(f'Average entropy: {mean(entropy_scores):.3f}')
print(f'Frequency anomaly=True: {frequency_anomaly_true:,}')
print(f'Sequence anomaly=True: {sequence_anomaly_true:,}')


## 10. Drill into domain-specific fields

The dataset mixes several security domains: network, firewall, identity, endpoint, cloud, IoT, IDS, and AI security. A complete analysis should inspect the important domain fields for each one.

This section summarizes the most common values in fields such as actions, protocols, alert types, categories, cloud services, and device types.

In [ ]:
action_counts = Counter(row['action'] for row in rows if not is_missing(row['action']))
protocol_counts = Counter(row['protocol'] for row in rows if not is_missing(row['protocol']))
alert_type_counts = Counter(row['alert_type'] for row in rows if not is_missing(row['alert_type']))
category_counts = Counter(row['category'] for row in rows if not is_missing(row['category']))
cloud_service_counts = Counter(row['cloud_service'] for row in rows if not is_missing(row['cloud_service']))
device_type_counts = Counter(row['device_type'] for row in rows if not is_missing(row['device_type']))

print('Top actions:')
for value, count in top_items(action_counts, 20):
    print(f'{value:<28} {count:>6,}')

print('\nTop protocols:')
for value, count in top_items(protocol_counts, 15):
    print(f'{value:<10} {count:>6,}')

print('\nTop IDS alert types:')
for value, count in top_items(alert_type_counts, 15):
    print(f'{value:<24} {count:>6,}')

print('\nTop IDS categories:')
for value, count in top_items(category_counts, 10):
    print(f'{value:<12} {count:>6,}')

print('\nCloud services:')
for value, count in cloud_service_counts.most_common():
    print(f'{value:<10} {count:>6,}')

print('\nIoT device types:')
for value, count in device_type_counts.most_common():
    print(f'{value:<12} {count:>6,}')


## 11. Find high-priority combinations for investigation

Counts alone are not enough for triage. This section looks for combinations that are likely to matter most to an analyst.

Examples:
- event types that generate the most `critical` or `emergency` findings
- sources contributing the most high-severity events
- common event and severity combinations that may dominate analyst workload

In [ ]:
high_severity_rows = [row for row in rows if row['severity'] in ('critical', 'emergency')]
high_severity_event_types = Counter(row['event_type'] for row in high_severity_rows)
high_severity_sources = Counter(row['source'] for row in high_severity_rows)
event_severity_pairs = Counter((row['event_type'], row['severity']) for row in rows)

print(f'High severity rows: {len(high_severity_rows):,} ({pct(len(high_severity_rows), len(rows)):.2f}%)')

print('\nHigh severity by event type:')
for value, count in high_severity_event_types.most_common():
    print(f'{value:<12} {count:>6,}')

print('\nHigh severity by source:')
for value, count in top_items(high_severity_sources, 15):
    print(f'{value:<28} {count:>6,}')

print('\nTop event + severity combinations:')
for (event_type, severity), count in event_severity_pairs.most_common(15):
    print(f'{event_type:<12} {severity:<10} {count:>6,}')


## 12. Final findings and analyst comments

The final section converts the raw statistics into comments that explain what the dataset is doing.

These comments are intentionally written like analyst notes so they can be reused in a report or presentation.

In [ ]:
comments = []

comments.append(f"1. The dataset contains {len(rows):,} events across {len(event_type_counts)} main event families, which makes it broad enough for cross-domain security analysis.")
comments.append("2. Missingness is largely structural, not random. Many fields are populated only for the event types where they make sense, such as `model_id` for AI events or `protocol` for network and firewall events.")
comments.append(f"3. The timestamp range runs from {min_ts} to {max_ts}. Most events are concentrated in 2025, but {len(out_of_2025):,} rows fall outside that year, so time-series conclusions should be made carefully.")
comments.append(f"4. Severity is fairly balanced overall, but there are {severity_counts['critical']:,} critical and {severity_counts['emergency']:,} emergency records, which is a material triage load.")
comments.append("5. `ids_alert` is the only event family producing emergency records in this dataset, which makes it a strong candidate for first-level escalation review.")
comments.append(f"6. `advanced_metadata` is consistently present and provides stable enrichment, with an average risk score of {mean(risk_scores):.2f} and average confidence of {mean(confidence_scores):.3f}.")
comments.append(f"7. `behavioral_analytics` is available for only {behavioral_rows:,} rows ({pct(behavioral_rows, len(rows)):.2f}%), so anomaly-based conclusions should be limited to that subset.")
comments.append("8. The action vocabulary shows that the dataset covers modern threat themes including prompt injection, training data poisoning, crypto mining, command injection, credential abuse, and endpoint persistence behavior.")
comments.append("9. High-severity records are spread across many vendors, so there is no single source dominating alert quality. That suggests the data is synthetic or intentionally balanced rather than organically collected from one environment.")
comments.append("10. As a next step, the best follow-on analyses would be event chaining by user or IP, outlier scoring on risk and confidence, and building event-type-specific detection dashboards.")

print('Analyst comments:')
for comment in comments:
    print(comment)


In [ ]:
import os
from env_loader import load_local_env

load_local_env(override=True)
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY', os.getenv('GEMINI_API_KEY', ''))
GEMINI_MODEL = os.getenv('GOOGLE_MODEL', os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'))

if GOOGLE_API_KEY:
    print(f'Gemini API key detected via .env/environment. Model: {GEMINI_MODEL}')
else:
    print('Gemini API key not set. Section 13 will use the offline heuristic fallback and still write the prompt/context artifacts.')


## Phase 6: Correlation Analysis

### Event Co-Occurrence Matrix

This section measures which event types appear together within the same minute for the same normalized source. The goal is to highlight short attack chains or multi-domain activity without requiring full sequence reconstruction.

The code groups events into `(source, minute)` buckets, counts unique event-type pairs inside each bucket, and then prints both the strongest co-occurring pairs and a compact matrix for the main event families.


In [ ]:
from itertools import combinations

# Group events into source + minute buckets so we can see which event families appear together
# in the same short time window. Missing source IPs are normalized to 'unknown'.
events_by_source_minute = defaultdict(list)

for row in rows:
    normalized_source = row['src_ip'] if not is_missing(row['src_ip']) else 'unknown'
    minute_bucket = row['parsed_timestamp'].strftime('%Y-%m-%d %H:%M')
    events_by_source_minute[(normalized_source, minute_bucket)].append(row['event_type'])

# Count unique event-type pairs per bucket. Using a set here prevents one noisy minute
# from dominating the result just because the same event type repeats many times.
event_pair_counts = Counter()
multi_event_buckets = 0

for event_list in events_by_source_minute.values():
    unique_event_types = sorted(set(event_list))
    if len(unique_event_types) < 2:
        continue

    multi_event_buckets += 1
    for pair in combinations(unique_event_types, 2):
        event_pair_counts[pair] += 1

# Build a symmetric matrix so the notebook reader can inspect pair strength by event family.
cooccurrence_matrix = defaultdict(dict)
for (left_event, right_event), count in event_pair_counts.items():
    cooccurrence_matrix[left_event][right_event] = count
    cooccurrence_matrix[right_event][left_event] = count

matrix_event_types = [name for name, _ in event_type_counts.most_common()]

print('Correlation method:')
print('- Same normalized source IP')
print('- Same minute-level timestamp bucket')
print('- Unique event-type pairs per bucket')
print(f'- Buckets with 2+ event families: {multi_event_buckets:,}')

print('\nTop co-occurring event pairs:')
for (left_event, right_event), count in event_pair_counts.most_common(15):
    print(f'{left_event:<12} + {right_event:<12} {count:>6,}')

print('\nEvent co-occurrence matrix (same source, same minute):')
header = 'event_type'.ljust(12) + ''.join(name[:10].rjust(12) for name in matrix_event_types)
print(header)
for left_event in matrix_event_types:
    row_text = left_event.ljust(12)
    for right_event in matrix_event_types:
        if left_event == right_event:
            row_text += '-'.rjust(12)
        else:
            row_text += f"{cooccurrence_matrix[left_event].get(right_event, 0):,}".rjust(12)
    print(row_text)


### Risk Matrix Calculation

This subsection converts the source-level severity distribution into a compact risk matrix. It shows how many events each normalized source contributes at each severity level and highlights sources where `critical` and `emergency` events make up a large share of the total.

The calculation is useful for prioritization because a source with moderate volume but a high concentration of severe events can be more urgent than a noisy source dominated by lower-severity traffic.

In [ ]:
# Build a source -> severity -> count matrix so we can compare how risky each source looks
# based on the distribution of its event severities.
risk_matrix = defaultdict(lambda: defaultdict(int))

for row in rows:
    normalized_source = row['src_ip'] if not is_missing(row['src_ip']) else 'unknown'
    severity = row['severity'].lower()
    risk_matrix[normalized_source][severity] += 1

# Summarize each source into analyst-friendly metrics. We keep both raw counts and percentages
# so the notebook can show volume-heavy sources and severity-heavy sources separately.
severity_order_matrix = ['emergency', 'critical', 'high', 'medium', 'low', 'info']
source_risk_metrics = []

for source, severities in risk_matrix.items():
    total_events = sum(severities.values())
    emergency_count = severities.get('emergency', 0)
    critical_count = severities.get('critical', 0)
    high_count = severities.get('high', 0)
    critical_plus_count = emergency_count + critical_count
    high_plus_count = critical_plus_count + high_count

    source_risk_metrics.append({
        'source': source,
        'total_events': total_events,
        'emergency': emergency_count,
        'critical': critical_count,
        'high': high_count,
        'critical_plus_count': critical_plus_count,
        'critical_plus_pct': pct(critical_plus_count, total_events),
        'high_plus_pct': pct(high_plus_count, total_events),
        'severity_breakdown': {level: severities.get(level, 0) for level in severity_order_matrix},
    })

by_total_events = sorted(source_risk_metrics, key=lambda item: item['total_events'], reverse=True)
by_critical_pct = sorted(
    source_risk_metrics,
    key=lambda item: (item['critical_plus_pct'], item['critical_plus_count'], item['total_events']),
    reverse=True,
)

print('Risk matrix summary:')
print('- Source is normalized to src_ip or unknown when src_ip is missing')
print('- critical_plus_pct = (critical + emergency) / total events')
print('- high_plus_pct = (high + critical + emergency) / total events')

print('\nTop sources by total event volume:')
print(f"{'source':<18} {'total':>8} {'critical+':>10} {'critical+%':>12} {'high+%':>10}")
for item in by_total_events[:10]:
    print(f"{item['source'][:18]:<18} {item['total_events']:>8,} {item['critical_plus_count']:>10,} {item['critical_plus_pct']:>11.2f}% {item['high_plus_pct']:>9.2f}%")

print('\nTop sources by critical/emergency concentration (minimum 25 events):')
print(f"{'source':<18} {'total':>8} {'emerg':>8} {'crit':>8} {'critical+%':>12} {'high+%':>10}")
for item in [entry for entry in by_critical_pct if entry['total_events'] >= 25][:10]:
    print(f"{item['source'][:18]:<18} {item['total_events']:>8,} {item['emergency']:>8,} {item['critical']:>8,} {item['critical_plus_pct']:>11.2f}% {item['high_plus_pct']:>9.2f}%")

print('\nRisk matrix for top 8 sources by volume:')
header = 'source'.ljust(18) + ''.join(level[:8].rjust(10) for level in severity_order_matrix)
print(header)
for item in by_total_events[:8]:
    row_text = item['source'][:18].ljust(18)
    for level in severity_order_matrix:
        row_text += f"{item['severity_breakdown'][level]:,}".rjust(10)
    print(row_text)


## Phase 7: Visualization Generation

### Python Matplotlib Integration and Detailed Visualization

This section converts the tabular results into reusable visual outputs. The goal is to make the earlier findings easier to inspect at a glance and easier to reuse in reports or slides.

The code below generates:
- an overview dashboard for event volume, severity, risk, and confidence
- a heatmap for the event co-occurrence matrix
- a heatmap for the source-level risk matrix

All plots are saved into a local `visualizations` folder so they can be reused outside the notebook.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Save all generated figures into a dedicated folder so the outputs are easy to reuse.
visualization_dir = Path('visualizations')
visualization_dir.mkdir(exist_ok=True)

# Use a consistent plotting style so every figure in this phase shares the same visual language.
plt.style.use('seaborn-v0_8-whitegrid')

# ------------------------------
# 1. Overview dashboard
# ------------------------------
# This dashboard combines four high-signal views into a single figure:
# - event-type distribution
# - severity distribution
# - risk-score histogram
# - confidence-score histogram
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('SIEM Analysis Overview Dashboard', fontsize=18, fontweight='bold')

top_events = event_type_counts.most_common()
event_labels = [name for name, _ in top_events]
event_values = [count for _, count in top_events]
axes[0, 0].barh(event_labels, event_values, color='#2F6BFF')
axes[0, 0].invert_yaxis()
axes[0, 0].set_title('Event Type Distribution')
axes[0, 0].set_xlabel('Count')

severity_order_plot = ['emergency', 'critical', 'high', 'medium', 'low', 'info']
severity_values = [severity_counts[level] for level in severity_order_plot]
severity_colors = ['#8B0000', '#D62728', '#FF7F0E', '#F2C14E', '#8BC34A', '#4FC3F7']
axes[0, 1].bar(severity_order_plot, severity_values, color=severity_colors)
axes[0, 1].set_title('Severity Distribution')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=25)

axes[1, 0].hist(risk_scores, bins=20, color='#7A3FFF', edgecolor='white')
axes[1, 0].axvline(mean(risk_scores), color='black', linestyle='--', linewidth=1.5, label=f"mean={mean(risk_scores):.2f}")
axes[1, 0].set_title('Risk Score Distribution')
axes[1, 0].set_xlabel('Risk Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

axes[1, 1].hist(confidence_scores, bins=20, color='#00A896', edgecolor='white')
axes[1, 1].axvline(mean(confidence_scores), color='black', linestyle='--', linewidth=1.5, label=f"mean={mean(confidence_scores):.3f}")
axes[1, 1].set_title('Confidence Score Distribution')
axes[1, 1].set_xlabel('Confidence Score')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

fig.tight_layout(rect=[0, 0, 1, 0.97])
overview_dashboard_path = visualization_dir / 'overview_dashboard.png'
fig.savefig(overview_dashboard_path, dpi=200, bbox_inches='tight')
plt.close(fig)

# ------------------------------
# 2. Event co-occurrence heatmap
# ------------------------------
# Reuse the correlation matrix built earlier and render it as a heatmap so dense pairings
# stand out faster than they do in plain text.
cooccurrence_array = np.array([
    [cooccurrence_matrix[left_event].get(right_event, 0) if left_event != right_event else 0 for right_event in matrix_event_types]
    for left_event in matrix_event_types
])

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(cooccurrence_array, cmap='YlOrRd')
ax.set_title('Event Co-Occurrence Heatmap')
ax.set_xticks(range(len(matrix_event_types)))
ax.set_yticks(range(len(matrix_event_types)))
ax.set_xticklabels(matrix_event_types, rotation=45, ha='right')
ax.set_yticklabels(matrix_event_types)
fig.colorbar(image, ax=ax, label='Same-minute pair count')

for row_index in range(len(matrix_event_types)):
    for col_index in range(len(matrix_event_types)):
        value = cooccurrence_array[row_index, col_index]
        if value:
            ax.text(col_index, row_index, f'{value:,}', ha='center', va='center', fontsize=8, color='black')

fig.tight_layout()
cooccurrence_heatmap_path = visualization_dir / 'event_cooccurrence_heatmap.png'
fig.savefig(cooccurrence_heatmap_path, dpi=200, bbox_inches='tight')
plt.close(fig)

# ------------------------------
# 3. Source risk matrix heatmap
# ------------------------------
# Focus on the highest-volume sources so the matrix stays readable while still surfacing
# the most operationally important severity patterns.
top_risk_sources = by_total_events[:8]
risk_array = np.array([
    [item['severity_breakdown'][level] for level in severity_order_matrix]
    for item in top_risk_sources
])
risk_source_labels = [item['source'][:18] for item in top_risk_sources]

fig, ax = plt.subplots(figsize=(11, 7))
image = ax.imshow(risk_array, cmap='Blues')
ax.set_title('Source Risk Matrix Heatmap')
ax.set_xticks(range(len(severity_order_matrix)))
ax.set_yticks(range(len(risk_source_labels)))
ax.set_xticklabels(severity_order_matrix, rotation=35, ha='right')
ax.set_yticklabels(risk_source_labels)
fig.colorbar(image, ax=ax, label='Event count')

for row_index in range(len(risk_source_labels)):
    for col_index in range(len(severity_order_matrix)):
        value = risk_array[row_index, col_index]
        if value:
            ax.text(col_index, row_index, f'{value:,}', ha='center', va='center', fontsize=8, color='black')

fig.tight_layout()
risk_heatmap_path = visualization_dir / 'source_risk_heatmap.png'
fig.savefig(risk_heatmap_path, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Saved visualization files:')
print(f'- {overview_dashboard_path}')
print(f'- {cooccurrence_heatmap_path}')
print(f'- {risk_heatmap_path}')


# SECTION 13: LLM-Based Threat Analysis with Prompt Engineering

This section turns the dataset into a prompt-engineered analyst packet and then runs a threat-analysis workflow.

The workflow uses four controls:
- **Role prompt**: force the model to behave like a senior SOC analyst
- **Metric rubric**: bind the model to the risk, confidence, severity, and z-score thresholds
- **Evidence packet**: provide only structured dataset context and representative high-priority cases
- **Structured output**: require JSON so findings, false-positive risk, and recommended actions are machine-readable

**Execution behavior**
- Keys are loaded from `.env` or the current shell environment
- If `GOOGLE_API_KEY` or `GEMINI_API_KEY` is set, the notebook uses Gemini structured JSON output
- If no Gemini key is set, the notebook falls back to an offline heuristic synthesis using the same metric rubric
- In both modes, the notebook writes the prompt, context packet, JSON output, and Markdown report to disk


## 13.1 Run Prompt-Engineered Threat Analysis

The code below builds the context packet from `advanced_siem.csv`, applies the prompt rubric, and writes these artifacts:
- `prompt_engineered_threat_context.json`
- `prompt_engineered_threat_prompt.txt`
- `prompt_engineered_threat_analysis.json`
- `PROMPT_ENGINEERED_THREAT_ANALYSIS.md`


In [ ]:
import importlib
import prompt_engineered_threat_analysis as _peta_mod
importlib.reload(_peta_mod)
from prompt_engineered_threat_analysis import run_analysis

llm_threat_analysis = run_analysis(
    data_path='advanced_siem.csv',
    output_dir='.',
    use_llm=bool(GOOGLE_API_KEY),
)

print(f"Analysis mode: {llm_threat_analysis['analysis_mode']}")
if llm_threat_analysis['llm_error']:
    print(f"LLM status: {llm_threat_analysis['llm_error']}")
print(f"Report: {llm_threat_analysis['report_path']}")
print(f"Prompt: {llm_threat_analysis['prompt_path']}")
print(f"Context: {llm_threat_analysis['context_path']}")
print()
print(llm_threat_analysis['analysis']['executive_summary'])
print()
print('Top findings:')
for finding in llm_threat_analysis['analysis']['priority_findings'][:3]:
    print(f"- {finding['finding']} | severity={finding['severity']} | confidence={finding['confidence_level']}")


## 13.2 Full-Dataset Incident Recommendations

The earlier LLM section summarizes the dataset and highlights top-priority cases, so it naturally produces fewer recommendations than the 100,000 raw SIEM rows. This step processes every event in `advanced_siem.csv`, generates a row-level recommendation for each one, and writes the full result set to CSV.


In [ ]:
from pathlib import Path
import importlib
import pandas as pd
from IPython.display import display

import prompt_engineered_threat_analysis as _peta_full
importlib.reload(_peta_full)
from prompt_engineered_threat_analysis import export_all_case_recommendations

# Process the entire 100,000-row SIEM dataset instead of the smaller priority-case subset.
# The helper prints progress every 5,000 rows so the notebook log shows long-running work.
all_case_export = export_all_case_recommendations(
    data_path='advanced_siem.csv',
    output_dir='llm_outputs',
    progress_every=5000,
    log_fn=print,
)

all_case_results_path = Path(all_case_export['csv_path'])
all_case_results_df = pd.read_csv(all_case_results_path)

print()
print(f"Saved full-dataset incident recommendations to: {all_case_results_path}")
print(f"Rows written: {len(all_case_results_df):,}")
print()
print('Classification counts:')
print(all_case_results_df['classification'].value_counts().to_string())
print()
print('Recommended owner counts:')
print(all_case_results_df['recommended_owner'].value_counts().head(10).to_string())
print()
preview_columns = [
    'event_id',
    'timestamp',
    'event_type',
    'severity',
    'classification',
    'recommended_owner',
    'detailed_classification_reason',
    'recommended_action',
]
display(all_case_results_df[preview_columns].head(10))


## 13.3 Visualize Full-Dataset Recommendations

This visualization uses the exported 100,000-row recommendation file so the charts reflect the full SIEM workload rather than only the summarized LLM findings.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# Load the full recommendation export and build a compact dashboard for review.
# The charts focus on analyst workload: classification, severity, owners, and top event types.
full_results_source = all_case_results_path if 'all_case_results_path' in globals() else Path('llm_outputs/all_siem_incident_recommendations.csv')
full_results_df = pd.read_csv(full_results_source)
viz_dir = Path('visualizations/llm_results')
viz_dir.mkdir(parents=True, exist_ok=True)
dashboard_path = viz_dir / 'all_siem_incident_recommendations_dashboard.png'

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

classification_counts = full_results_df['classification'].value_counts()
classification_counts.plot(kind='bar', ax=axes[0, 0], color=['#b2182b', '#ef8a62', '#67a9cf', '#2166ac'])
axes[0, 0].set_title('Classification Distribution')
axes[0, 0].set_xlabel('Classification')
axes[0, 0].set_ylabel('Events')
axes[0, 0].tick_params(axis='x', rotation=20)

severity_order = ['emergency', 'critical', 'high', 'medium', 'low', 'info']
severity_counts = full_results_df['severity'].value_counts().reindex(severity_order, fill_value=0)
severity_counts.plot(kind='bar', ax=axes[0, 1], color='#4c78a8')
axes[0, 1].set_title('Severity Distribution')
axes[0, 1].set_xlabel('Severity')
axes[0, 1].set_ylabel('Events')
axes[0, 1].tick_params(axis='x', rotation=20)

owner_counts = full_results_df['recommended_owner'].value_counts().head(8).sort_values()
owner_counts.plot(kind='barh', ax=axes[1, 0], color='#72b7b2')
axes[1, 0].set_title('Top Recommended Owners')
axes[1, 0].set_xlabel('Events')
axes[1, 0].set_ylabel('Owner')

top_event_types = full_results_df['event_type'].value_counts().head(10).sort_values()
top_event_types.plot(kind='barh', ax=axes[1, 1], color='#f58518')
axes[1, 1].set_title('Top Event Types in Full Recommendation Export')
axes[1, 1].set_xlabel('Events')
axes[1, 1].set_ylabel('Event Type')

fig.suptitle('Full-Dataset SIEM Recommendation Dashboard', fontsize=16)
fig.tight_layout()
fig.savefig(dashboard_path, dpi=160, bbox_inches='tight')
plt.show()
print(f"Saved dashboard to: {dashboard_path}")
